# Práctica guiada · Ajuste de hiperparámetros con tres clasificadores
### Regresión logística · SVM · Naive Bayes

**FP13 · Inteligencia Artificial** —  Universidad Autónoma de Guadalajara
**Unidad 5:** 5.5 Regresión logística · 5.6 Naive Bayes · 5.7 Matrices de confusión · 5.8 SVM — con adelanto de **6.5 Búsqueda de rejilla**

---

### 🏥 El problema clínico
Un servicio de cardiología quiere una herramienta de **apoyo al triaje**: con 11 datos clínicos de rutina (edad, tipo de dolor torácico, colesterol, ECG en reposo, frecuencia cardiaca máxima, depresión del segmento ST…) debe decidir si un paciente **probablemente tiene enfermedad cardiaca** y conviene enviarlo a estudios especializados.

In [ ]:
# Importamos librerias base para el script
import warnings
warnings.filterwarnings("ignore") # Para evitar que salgan alertas rojas en la terminal

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import kagglehub 

# Importando modulos de scikit-learn para el modelo
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay, confusion_matrix,
                             accuracy_score, recall_score, precision_score, f1_score)

---
## 1 · Los datos

**Heart Failure Prediction Dataset** (fedesoriano, Kaggle): 918 pacientes de cinco cohortes cardiológicas.

| Variable | Significado clínico |
|---|---|
| `Age` | Edad (años) |
| `Sex` | Sexo (M / F) |
| `ChestPainType` | Dolor torácico: TA angina típica · ATA atípica · NAP no anginoso · ASY asintomático |
| `RestingBP` | Presión arterial sistólica en reposo (mm Hg) |
| `Cholesterol` | Colesterol sérico (mg/dL) |
| `FastingBS` | Glucosa en ayuno > 120 mg/dL (1 = sí) |
| `RestingECG` | ECG en reposo: Normal · ST (anomalía ST-T) · LVH (hipertrofia ventricular izquierda) |
| `MaxHR` | Frecuencia cardiaca máxima en prueba de esfuerzo |
| `ExerciseAngina` | Angina inducida por el ejercicio (Y / N) |
| `Oldpeak` | Depresión del segmento ST con el esfuerzo (mm) |
| `ST_Slope` | Pendiente del ST en el pico del esfuerzo: Up · Flat · Down |
| **`HeartDisease`** | **Lo que queremos predecir: 1 = con enfermedad cardiaca · 0 = sin enfermedad** |

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
# Descargando el dataset de Kaggle para empezar a trabajar
datos_corazon = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS,
                                       "fedesoriano/heart-failure-prediction", "heart.csv")

In [ ]:
datos_corazon.head()

In [ ]:
# Imprimiendo el tamaño de la matriz de datos (renglones y columnas)
print("Shape del dataframe:", datos_corazon.shape)

---
## 2 · Preparar los datos

Tres decisiones, todas vistas en la Unidad 4:
1. **Valores imposibles → faltantes.** Nadie vivo tiene colesterol 0 ni presión arterial 0: son celdas que nadie llenó.
2. **Separar `X` de `y`.** `X` = lo que el modelo puede ver; `y` = la respuesta correcta.
3. **One-hot.** Los modelos solo entienden números, así que cada columna de texto se convierte en columnas de 0 y 1.

In [ ]:
# Verificando los tipos de variables del sistema
datos_corazon.dtypes

In [ ]:
# Buscando valores vacios o nulos en los registros
datos_corazon.isna().sum()

In [ ]:
# Resumen estadistico de las variables numericas del dataset
datos_corazon.describe().T

In [ ]:
# Ajustando mediciones anomalas (colesterol o presion en cero es un error del sensor/medicion)
datos_corazon['Cholesterol'] = datos_corazon['Cholesterol'].replace(0, np.nan)
datos_corazon['RestingBP'] = datos_corazon['RestingBP'].replace(0, np.nan)

In [ ]:
# Comprobamos como quedaron los nulos despues del ajuste
datos_corazon.isna().sum()

# Dividiendo los datos

In [ ]:
# Definimos nuestras entradas X y nuestra salida a predecir y
salida_y = datos_corazon['HeartDisease']
entradas_x = datos_corazon.drop(columns=['HeartDisease'])

In [ ]:
# Separando las columnas categoricas
cols_categoricas = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

# Iteramos para ver que categorias tiene cada una
for columna in cols_categoricas:
    print(f"--- {columna} ---")
    print(entradas_x[columna].value_counts())
    print("\n")

In [ ]:
print(entradas_x.head())
print(entradas_x.shape)

In [ ]:
# Codificamos las variables de texto a binario (One-Hot) para poder procesarlas matematicamente
entradas_x = pd.get_dummies(entradas_x, drop_first=True, dtype=int)

# Revisamos como quedo la matriz
print(entradas_x.head())
print(entradas_x.shape)

# Haciendo el split de los datos

In [ ]:
# Analizando la distribucion de la variable de salida
print(datos_corazon['HeartDisease'].value_counts(normalize=True))

prop_enfermos = datos_corazon['HeartDisease'].value_counts(normalize=True)[1]

print('\n')
print(f'Proporcion: Tenemos un {prop_enfermos:.2%} de casos positivos. Esta balanceado, pero usaremos stratify para mayor seguridad en el particionamiento.')

In [ ]:
# Particionando datos para entrenamiento y validacion
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(entradas_x, salida_y, test_size=0.2, stratify=salida_y, random_state=42)

print(f'Matriz de entrenamiento X: {X_entrenamiento.shape}')
print(f'Matriz de prueba X: {X_prueba.shape}')

---
## 3 · Repaso: ¿cómo vamos a evaluar?

Las filas son la **realidad** y las columnas lo que **predice** el modelo:

| | Predice **Sano** | Predice **Enfermo** |
|---|:---:|:---:|
| **Realmente sano** | ✅ VN · sano bien descartado | ⚠️ FP · falsa alarma |
| **Realmente enfermo** | ❌ **FN · enfermo que se va a casa** | ✅ VP · enfermo detectado |

<br>

| Métrica | Pregunta clínica que responde | Nombre en `scoring` |
|---|---|---|
| **Exactitud** | De cada 100 pacientes, ¿cuántos clasifica bien? | `"accuracy"` |
| **Precisión** | De cada 100 que el modelo marca como enfermos, ¿cuántos realmente lo están? | `"precision"` |
| **Sensibilidad** | De cada 100 enfermos, ¿a cuántos detecta? | `"recall"` |
| **F1** | Un solo número que exige **a la vez** buena precisión y buena sensibilidad | `"f1"` |
| **AUC** | Si tomo un enfermo y un sano al azar, ¿qué tan seguido el modelo le asigna más riesgo al enfermo? | `"roc_auc"` |
| **Especificidad** | De cada 100 sanos, ¿a cuántos descarta correctamente? | *(es el recall de la clase "Sano")* |

---
# Regresion Logistica

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer

# Seleccionando parametros a normalizar
vars_numericas = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]

# Bloque de preprocesamiento (imputacion y escalado de señales)
preprocesador = ColumnTransformer( 
    transformers=[
        ("num", Pipeline([
            ("imputar", SimpleImputer(strategy='median')), # Llenar huecos con la mediana
            ('escalar', StandardScaler()),
             ]), vars_numericas),
    ],
     remainder='passthrough',
    verbose_feature_names_out=False
)

# Creando el pipeline de la Regresion Logistica
pipe_logistica = Pipeline([
    ("prep", preprocesador),
    ('modelo', LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0, 
                                  max_iter=5000, random_state=42))
])

# Entrenando el sistema
pipe_logistica.fit(X_entrenamiento, y_entrenamiento)

In [ ]:
nombres_clases = ["Sano", "Enfermo"]
lista_metricas = ["accuracy", "precision", "recall", "f1", "roc_auc"]

# Generando predicciones de prueba
pred_logistica = pipe_logistica.predict(X_prueba)

# Evaluacion del desempeño
print(classification_report(y_prueba, pred_logistica, target_names=nombres_clases))

# Matriz de confusion
ConfusionMatrixDisplay.from_predictions(y_prueba, pred_logistica, display_labels=nombres_clases, cmap="Blues")
plt.title("Matriz de Confusión - Reg. Logística Base")
plt.show()

In [ ]:
grid_reglog = {
    "modelo__C":            [0.001, 0.01, 0.1, 1, 10, 100],
    "modelo__l1_ratio":     [0, 0.5, 1],             # Balance regularizacion
    "modelo__class_weight": [None, "balanced"],
}

In [ ]:
top_modelos_log = {}

# Barriendo la red de hiperparametros para cada metrica
for m in lista_metricas:
    busqueda = GridSearchCV(pipe_logistica, grid_reglog, scoring=m, cv=5, n_jobs=-1)
    busqueda.fit(X_entrenamiento, y_entrenamiento)
    top_modelos_log[m] = busqueda.best_estimator_
    print(f"{m} >> {round(busqueda.best_score_, 3)} >> {busqueda.best_params_}")

**Justificación de métrica seleccionada:**
Elegí el modelo optimizado por precisión (precision) porque en el diseño de un sistema de diagnóstico real, los falsos positivos cuestan caro. Si por error nuestro algoritmo diagnostica a un paciente sano como enfermo, el hospital desperdiciará recursos, tiempo y dinero en estudios especializados innecesarios. Desde el punto de vista de control y eficiencia, es preferible un sistema sumamente preciso que minimice las falsas alarmas.

# Mquinas de vectores de Soporte (SVM)

In [ ]:
from sklearn.svm import SVC

# Diseñando el pipeline para la Máquina de Vectores de Soporte (SVM)
# Añadí el random_state para estandarizar los resultados y hacer la ejecución reproducible
pipe_svm = Pipeline([
    ("prep", preprocesador),
    ('modelo', SVC(random_state=42))
])

# Entrenando el modelo SVM (Corrigiendo el error donde se llamaba a la regresion logistica)
pipe_svm.fit(X_entrenamiento, y_entrenamiento)

In [ ]:
# Validando la predicción generada por el SVM
pred_svm = pipe_svm.predict(X_prueba)

print("--- Desempeño del Modelo SVM ---")
print(classification_report(y_prueba, pred_svm, target_names=nombres_clases))

# Generando gráfica de matriz de confusión (en verde para distinguir del modelo logístico)
ConfusionMatrixDisplay.from_predictions(y_prueba, pred_svm, display_labels=nombres_clases, cmap="Greens")
plt.title("Matriz de Confusión - SVM")
plt.show()

**Corrección en el bloque SVM:**
En la celda anterior se ajustó el entrenamiento para utilizar la variable `pipe_svm.fit()` en lugar del modelo logístico original que estaba copiado por error. Adicionalmente, implementé el reporte de métricas y la respectiva matriz de confusión para cerrar el bloque, logrando así validar el rendimiento técnico y hacer la comparativa final.